# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [ ]:
## Environment setup

# install piper-sample-generator (currently only supports linux systems)
!git clone https://github.com/dscripka/piper-sample-generator  # NOT rhasspy/piper-sample-generator -- that upstream repo lacks the root-level generate_samples.py this fork provides, which train.py imports directly
!wget -O piper-sample-generator/models/en-us-libritts-high.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt'  # matches generate_samples.py's default model= path; the old v2.0.0/en_US-libritts_r-medium.pt name doesn't
!sed -i 's/model = torch.load(model_path)/model = torch.load(model_path, weights_only=False)/' piper-sample-generator/generate_samples.py  # PyTorch 2.6+ defaults torch.load to weights_only=True, which blocks this trusted checkpoint's custom class
!pip install piper-phonemize
!pip install webrtcvad
!pip install "onnxruntime>=1.10.0,<2"  # openwakeword needs this; the notebook never installed it explicitly
!pip install onnxscript  # current torch.onnx.export unconditionally imports its dynamo-based exporter internals, which need this
!apt-get update -qq
!apt-get -qq -y install espeak-ng libespeak-ng1
!ldconfig
!pip install espeak-phonemizer  # generate_samples.py (dscripka fork) imports this -- distinct from piper-phonemize above

# install openwakeword (full installation to support training)
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword
!cd openwakeword

# install other dependencies
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install -U "torch_audiomentations>=0.12,<1"  # 0.11.0 calls torchaudio.set_audio_backend(), removed in current Colab torchaudio
!pip install acoustics==0.2.6
!pip install tensorflow-cpu==2.8.1
!pip install tensorflow_probability==0.16.0
!pip install onnx_tf==1.10.0
!pip install pronouncing==0.2.0
!pip install -U "datasets>=2.19,<3"  # unpinned: 2.14.6 pulls in pyarrow==11.0.0, which has no wheel for current Colab Python and fails to build from source
!pip install deep-phonemizer==0.0.19

# Download required models (workaround for Colab)
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite


In [ ]:
# torch_audiomentations==0.12.0 still calls torchaudio.info()/torchaudio.load(), both removed as
# top-level functions in current torchaudio -- patch its IO helper to use soundfile instead.
!pip install soundfile

_tam_io_path = "/usr/local/lib/python3.13/dist-packages/torch_audiomentations/utils/io.py"
with open(_tam_io_path) as f:
    _src = f.read()

_old_metadata = '''    @staticmethod
    def get_audio_metadata(file_path: Union[str, Path]) -> tuple:
        """Return (num_samples, sample_rate)."""
        info = torchaudio.info(str(file_path))
        # Deal with backwards-incompatible signature change.
        # See https://github.com/pytorch/audio/issues/903 for more information.
        if type(info) is tuple:
            si, ei = info
            num_samples = si.length
            sample_rate = si.rate
        else:
            num_samples = info.num_frames
            sample_rate = info.sample_rate
        return num_samples, sample_rate'''

_new_metadata = '''    @staticmethod
    def get_audio_metadata(file_path: Union[str, Path]) -> tuple:
        """Return (num_samples, sample_rate)."""
        import soundfile as sf
        info = sf.info(str(file_path))
        return info.frames, info.samplerate'''

assert _old_metadata in _src, "get_audio_metadata block not found -- check installed torch_audiomentations version"
_src = _src.replace(_old_metadata, _new_metadata)

_old_load = '''            try:
                original_data, _ = torchaudio.load(
                    audio_path,
                    frame_offset=original_sample_offset,
                    num_frames=original_num_samples,
                )
            except TypeError:
                raise Exception(
                    "It looks like you are using an unsupported version of torchaudio."
                    " If you have 0.6 or older, please upgrade to a newer version."
                )'''

_new_load = '''            import soundfile as sf
            _sf_data, _ = sf.read(
                audio_path,
                start=original_sample_offset,
                frames=original_num_samples,
                dtype="float32",
                always_2d=True,
            )
            original_data = torch.from_numpy(_sf_data.T)'''

assert _old_load in _src, "torchaudio.load block not found -- check installed torch_audiomentations version"
_src = _src.replace(_old_load, _new_load)

with open(_tam_io_path, "w") as f:
    f.write(_src)
print("patched torch_audiomentations io.py")


In [ ]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm

# Make the editable-installed openwakeword package importable both here AND in the
# subprocesses spawned by "!{sys.executable} openwakeword/openwakeword/train.py ..." below --
# sys.path.append() only affects this notebook's own process, not those subprocesses, since
# each gets a fresh interpreter with its own sys.path. os.environ IS inherited by subprocesses.
sys.path.append("./openwakeword")
os.environ["PYTHONPATH"] = os.path.abspath("./openwakeword") + ":" + os.environ.get("PYTHONPATH", "")


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [ ]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [ ]:
## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -O {out_dir} {link}
!cd audioset && tar -xvf bal_train09.tar

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

# Convert audioset files to 16khz sample rate
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# The upstream notebook also downloads 1 hour of the Free Music Archive dataset ("rudraml/fma")
# here as extra background audio. That is a script-based dataset served over HTTP, and current
# `datasets` fails on it in streaming mode ("ValueError: Cannot seek streaming HTTP file"), so
# it's dropped: the training config below only lists './audioset_16k' in background_paths.


In [ ]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [ ]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config

In [ ]:
# Modify values in the config and save a new version -- customized for Manuel-mvp's wake word.
#
# The wake word is "anita", not "manuel": this pipeline generates its training clips with an
# *English* TTS voice (piper en-us-libritts-high), so the model learns the English rendering of
# the phrase. "Manuel" comes out as "man-WELL", which is not what a Spanish speaker says, and
# the first model missed real speech often. "Anita" is pronounced the same in both languages
# (ah-NEE-tah), has three syllables (openWakeWord detects longer phrases more reliably), and
# is not a word that comes up in classroom speech. The app loads the result as
# app/src/main/assets/wakeword/<model_name>.onnx (see WakeWordListener.DEFAULT_MODEL_ASSET_PATH).

config["target_phrase"] = ["anita"]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = 10000
config["n_samples_val"] = 2000
config["steps"] = 30000
config["target_accuracy"] = 0.7
config["target_recall"] = 0.5

# NOTE: the upstream notebook also lists './fma' here, but no cell in this notebook downloads
# that dataset -- only './audioset_16k' (cell above) actually exists on disk, so './fma' is
# dropped here to avoid a missing-directory error during training.
config["background_paths"] = ['./audioset_16k']
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    documents = yaml.dump(config, file)


# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [ ]:
# Step 1: Generate synthetic clips
# For the number of clips we are using, this should take ~10 minutes on a free Google Colab instance with a T4 GPU
# If generation fails, you can simply run this command again as it will continue generating until the
# number of files meets the targets specified in the config file

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

In [ ]:
# Step 2: Augment the generated clips

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

In [ ]:
# Current torch.onnx.export() defaults to its newer dynamo-based exporter, which silently
# produced a degenerate ~14KB model (missing weights) for this custom Model class. Force the
# legacy tracer (dynamo=False) and opset_version=17 (>=17 needed for LayerNormalization --
# opset_version=13 fails when the exporter tries to downgrade to it).
_train_py_path = "/content/openwakeword/openwakeword/train.py"
with open(_train_py_path) as f:
    _src = f.read()

_old = '''        torch.onnx.export(model_to_save.to("cpu"), torch.rand(self.input_shape)[None, ],
                          os.path.join(output_dir, model_name + ".onnx"), opset_version=13)'''
_new = '''        torch.onnx.export(model_to_save.to("cpu"), torch.rand(self.input_shape)[None, ],
                          os.path.join(output_dir, model_name + ".onnx"), opset_version=17, dynamo=False)'''

if _old in _src:
    _src = _src.replace(_old, _new)
    with open(_train_py_path, "w") as f:
        f.write(_src)
    print("patched train.py export_model() call")
elif _new in _src:
    print("train.py already patched")
else:
    raise AssertionError("export_model block not found -- train.py content differs from expected")


In [ ]:
# Step 3: Train model

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

In [ ]:
# Step 4 (Optional): On Google Colab, sometimes the .tflite model isn't saved correctly
# If so, run this cell to retry

# Manually save to tflite as this doesn't work right in colab
def convert_onnx_to_tflite(onnx_model_path, output_path):
    """Converts an ONNX version of an openwakeword model to the Tensorflow tflite format."""
    # imports
    import onnx
    import logging
    import tempfile
    from onnx_tf.backend import prepare
    import tensorflow as tf

    # Convert to tflite from onnx model
    onnx_model = onnx.load(onnx_model_path)
    tf_rep = prepare(onnx_model, device="CPU")
    with tempfile.TemporaryDirectory() as tmp_dir:
        tf_rep.export_graph(os.path.join(tmp_dir, "tf_model"))
        converter = tf.lite.TFLiteConverter.from_saved_model(os.path.join(tmp_dir, "tf_model"))
        tflite_model = converter.convert()

        logging.info(f"####\nSaving tflite mode to '{output_path}'")
        with open(output_path, 'wb') as f:
            f.write(tflite_model)

    return None

convert_onnx_to_tflite(f"my_custom_model/{config['model_name']}.onnx", f"my_custom_model/{config['model_name']}.tflite")


After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!